<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/14_GES_Aware_Genomic_RAG_Cell_7C7_Blinded_Evaluation_Packet_Authorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {ROOT}')
print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


In [2]:
from __future__ import annotations
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv, hashlib, json, re, tempfile
import pandas as pd
import pyarrow.parquet as pq

NOTEBOOK_NAME = '14_GES_Aware_Genomic_RAG_Cell_7C7_Blinded_Evaluation_Packet_Authorization.ipynb'
CELL_ID = '7C7'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1440
EXPECTED_QUESTIONS = 80
EXPECTED_RUBRICS = 640

EXPECTED_CELL_7C6_V2_TERMINAL_DECISION = (
    'PASS_STAGE7C6_V2_1440_FROZEN_SCORE_BLIND_LLM_GENERATION_REQUESTS_COMPLETED_'
    'USING_CELL7C5R_API_COMPATIBLE_STRUCTURED_OUTPUT_SCHEMA_WITH_ORIGINAL_SCIENTIFIC_'
    'EVIDENCE_ID_UNIQUENESS_VALIDATION_PRESERVED_RAW_AND_STRUCTURED_RESPONSES_'
    'MATERIALIZED_CHECKSUM_PROTECTED_CHECKPOINT_RESUMABLE_NO_SCORE_BEARING_AUDIT_'
    'CELL7A3_SCORES_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS_NEXT_EXECUTION_NOT_AUTHORIZED'
)

# Cell 7C6 V2
EXEC7C6 = ROOT/'outputs'/'rag_execution'/'stage7_rag'/'cell_7c6_exact_llm_generation_v2'
QC7C6 = ROOT/'outputs'/'quality_checks'/'stage7_rag'/'cell_7c6_exact_llm_generation_v2'
CFG7C6 = ROOT/'configs'/'stage7_rag'/'cell_7c6_exact_llm_generation_v2'

CELL_7C6 = OrderedDict([
    ('raw_response_envelopes', (EXEC7C6/'cell_7c6_v2_raw_response_envelopes_v1.jsonl', 'f6535fe7411980472d3b465fd3951e53f9259144255d74f0c2abb6a368aefca7')),
    ('response_inventory', (EXEC7C6/'cell_7c6_v2_generation_response_inventory_v1.parquet', '4a05de4b37a92eefbdde8673ec7f1f3a30c23ee63461a1c26dbb4a117296f13a')),
    ('structured_outputs', (EXEC7C6/'cell_7c6_v2_structured_response_outputs_v1.parquet', '267edad192f41ab32e211ee43b083ca167e6eb6f9d3e63008822e39b57317a46')),
    ('input_inventory', (CFG7C6/'cell_7c6_v2_verified_generation_input_inventory_v1.csv', 'c11e9ec0c8046925b56a9c71517e9ea729d9722177849c82ada4e2aa4f45d114')),
    ('execution_report', (QC7C6/'cell_7c6_v2_llm_generation_execution_report_v1.json', '56b56f9400e8ba7a4295c538718c9b0029b812aa8ab67d4bb54d8d44e6265e81')),
    ('qc', (QC7C6/'cell_7c6_v2_llm_generation_qc_v1.json', '62de169eec48993664025b71a2acf3e8fa2742a819039c0cddccd9ab85e40652')),
    ('manifest', (CFG7C6/'cell_7c6_v2_exact_llm_generation_manifest_v1.json', '9a84701858829d650d826109e32b3708f4b0c5d7b2ab393dc3f3d3afd5cc8293')),
])

CELL_7C4_CONTEXT = ('cell_7c4_score_blind_prompt_context_inventory_v1.parquet',
                    'd23b604dac645c158cd1b95cc9cd564fb6b9447279756b6cae589b5572dd4623')
CELL_7B3_ANSWER = ('cell_7b3_structured_answer_keys_v1.parquet',
                   'bccf3691336ed8e21a4e7b2876fcf6883ee9a8447938a0edf61d41fc5e0d0332')
CELL_7B3_RUBRIC = ('cell_7b3_question_rubric_assignment_inventory_v1.csv',
                   '4185ad173c5e3e2ecea31268b8d833ba111839f5ddd1b96c3fda083e01afcba9')
CELL_7B3_MANIFEST = ('cell_7b3_score_blind_materialization_manifest_v1.json',
                     '9d0674014f6650cde12f96abb75c06215c36870f7ea2af0dec1916e2b756a2c5')

H_7B2_MATERIALIZATION = '631b51556b97b3cf0db6d0989fa380b8fe3b226472de90c89f57eb0cae527da2'
H_7B2_ADJUDICATION = 'f79ed83b5e5a614815919f6d2b10efd01947c1512b02502e258820431a4c3ef0'
H_7B2_MANIFEST = 'a8a6379686bee22d347dd8f61c152d22a0d1c09cc6d74839ffe6d2eb86fc75c5'

AUTH_DIR = ROOT/'configs'/'stage7_rag'/'cell_7c7_blinded_evaluation_packet_authorization_v1'
QC_DIR = ROOT/'outputs'/'quality_checks'/'stage7_rag'/'cell_7c7_blinded_evaluation_packet_authorization_v1'
AUTH_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

OUTPUTS = OrderedDict([
    ('authorization', AUTH_DIR/'cell_7c7_stage7c_cell7c8_blinded_evaluation_packet_authorization_v1.json'),
    ('input_inventory', AUTH_DIR/'cell_7c7_authorized_blinded_evaluation_input_inventory_v1.csv'),
    ('protocol_snapshot', AUTH_DIR/'cell_7c7_frozen_blinded_evaluation_protocol_snapshot_v1.json'),
    ('qc', QC_DIR/'cell_7c7_blinded_evaluation_packet_authorization_qc_v1.json'),
    ('manifest', AUTH_DIR/'cell_7c7_blinded_evaluation_packet_authorization_manifest_v1.json'),
])

existing = [str(p) for p in OUTPUTS.values() if p.exists()]
if existing:
    raise FileExistsError('Cell 7C7 overwrite protection active:\\n- ' + '\\n- '.join(existing))

print(f'Authorization directory: {AUTH_DIR}')
print(f'QC directory           : {QC_DIR}')

Authorization directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c7_blinded_evaluation_packet_authorization_v1
QC directory           : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c7_blinded_evaluation_packet_authorization_v1


In [3]:
def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(chunk_size), b''):
            h.update(block)
    return h.hexdigest()

def sidecar_path(path: Path):
    return path.with_name(path.name + '.sha256')

def read_sidecar_hash(path: Path):
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty sidecar: {path}')
    token = text.split()[0]
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid sidecar format: {path}')
    return token.lower()

def sidecar_is_valid(path: Path):
    return path.exists() and sidecar_path(path).exists() and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)

def verify(label, path, expected):
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(f'{label} hash mismatch\\nExpected {expected}\\nObserved {observed}')
    if not sidecar_is_valid(path):
        raise AssertionError(f'{label} sidecar invalid')
    return {'input_id': label, 'path': str(path), 'sha256': observed,
            'bytes': int(path.stat().st_size), 'sidecar_path': str(sidecar_path(path)),
            'sidecar_valid': True}

def locate(filename, expected):
    candidates = [p for p in ROOT.rglob(filename) if p.is_file()]
    matches = [p for p in candidates if sha256_file(p) == expected]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename} with frozen hash; found {len(matches)}')
    return matches[0]

def locate_config_hash(expected):
    candidates = []
    for base in [ROOT/'configs', ROOT/'outputs'/'quality_checks']:
        if base.exists():
            candidates += [p for p in base.rglob('*') if p.is_file() and not p.name.endswith('.sha256')]
    matches = [p for p in candidates if sha256_file(p) == expected]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one config/protocol artifact for hash {expected}; found {len(matches)}')
    return matches[0]

def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def parquet_rows(path):
    return int(pq.ParquetFile(path).metadata.num_rows)

def csv_rows(path):
    with path.open('r', encoding='utf-8', newline='') as f:
        r = csv.reader(f); next(r)
        return sum(1 for _ in r)

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)+chr(10), encoding='utf-8')
    return sha256_file(path)

def write_csv(path, df):
    df.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)

def write_sidecar(path):
    sidecar_path(path).write_text(f'{sha256_file(path)}  {path.name}'+chr(10), encoding='utf-8')

with tempfile.TemporaryDirectory() as td:
    p = Path(td)/'x.json'
    write_json(p, {'ok': True}); write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


In [4]:
verified_inputs = []

for key, (path, expected) in CELL_7C6.items():
    rec = verify(f'cell_7c6_v2_{key}', path, expected)
    rec['source_cell'] = '7C6_V2'
    rec['row_level_content_opened_in_cell_7c7'] = False
    verified_inputs.append(rec)

manifest_7c6 = load_json(CELL_7C6['manifest'][0])
qc_7c6 = load_json(CELL_7C6['qc'][0])
report_7c6 = load_json(CELL_7C6['execution_report'][0])

assert manifest_7c6.get('terminal_decision') == EXPECTED_CELL_7C6_V2_TERMINAL_DECISION
assert manifest_7c6.get('next_authorized_cell') is None
assert manifest_7c6.get('answer_key_access_authorized') is False
assert manifest_7c6.get('evaluation_authorized') is False
assert int(qc_7c6.get('failed_checks', -1)) == 0
assert parquet_rows(CELL_7C6['structured_outputs'][0]) == 1440
assert parquet_rows(CELL_7C6['response_inventory'][0]) == 1440

acct = report_7c6.get('execution_accounting', {})
assert int(acct.get('structured_valid_responses', -1)) == 1440
assert int(acct.get('refusal_responses', -1)) == 0
assert int(acct.get('incomplete_responses', -1)) == 0
assert int(acct.get('json_parse_error_responses', -1)) == 0

print('Cell 7C6 V2 package                : 7/7 exact hashes + sidecars')
print('Cell 7C6 V2 terminal PASS         : VERIFIED')
print('Structured-valid responses        : 1,440 / 1,440')
print('Answer-key row content opened     : NO')
print('Condition identities unblinded    : NO')
print('RAG metrics calculated            : NO')

Cell 7C6 V2 package                : 7/7 exact hashes + sidecars
Cell 7C6 V2 terminal PASS         : VERIFIED
Structured-valid responses        : 1,440 / 1,440
Answer-key row content opened     : NO
Condition identities unblinded    : NO
RAG metrics calculated            : NO


In [5]:
context_path = locate(*CELL_7C4_CONTEXT)
answer_path = locate(*CELL_7B3_ANSWER)
rubric_path = locate(*CELL_7B3_RUBRIC)
manifest7b3_path = locate(*CELL_7B3_MANIFEST)

mat_protocol_path = locate_config_hash(H_7B2_MATERIALIZATION)
adj_protocol_path = locate_config_hash(H_7B2_ADJUDICATION)
manifest7b2_path = locate_config_hash(H_7B2_MANIFEST)

for label, path, expected, source in [
    ('cell_7c4_context_inventory', context_path, CELL_7C4_CONTEXT[1], '7C4'),
    ('cell_7b3_structured_answer_keys', answer_path, CELL_7B3_ANSWER[1], '7B3'),
    ('cell_7b3_rubric_assignments', rubric_path, CELL_7B3_RUBRIC[1], '7B3'),
    ('cell_7b3_manifest', manifest7b3_path, CELL_7B3_MANIFEST[1], '7B3'),
    ('cell_7b2_materialization_protocol', mat_protocol_path, H_7B2_MATERIALIZATION, '7B2'),
    ('cell_7b2_adjudication_protocol', adj_protocol_path, H_7B2_ADJUDICATION, '7B2'),
    ('cell_7b2_manifest', manifest7b2_path, H_7B2_MANIFEST, '7B2'),
]:
    rec = verify(label, path, expected)
    rec['source_cell'] = source
    rec['row_level_content_opened_in_cell_7c7'] = (source == '7B2')
    verified_inputs.append(rec)

answer_rows = parquet_rows(answer_path)
rubric_rows = csv_rows(rubric_path)
context_rows = parquet_rows(context_path)

assert answer_rows == 80
assert rubric_rows == 640
assert context_rows == 2400

materialization_protocol = load_json(mat_protocol_path)
adjudication_protocol = load_json(adj_protocol_path)

protocol_text = json.dumps(
    {'materialization': materialization_protocol, 'adjudication': adjudication_protocol},
    sort_keys=True, ensure_ascii=False
).lower()

for signal in ['atomic', 'correct', 'citation', 'reviewer', 'adjud']:
    if signal not in protocol_text:
        raise AssertionError(f'Expected frozen protocol signal not found: {signal}')

print(f'Structured answer keys              : {answer_rows} — hash VERIFIED')
print(f'Rubric assignments                  : {rubric_rows} — hash VERIFIED')
print(f'Score-blind context rows            : {context_rows:,} — hash VERIFIED')
print('Cell 7B2 materialization protocol   : VERIFIED')
print('Cell 7B2 adjudication protocol      : VERIFIED')
print('Answer-key row content opened       : NO')
print('Rubric row content opened           : NO')
print('Alias-to-condition mapping opened   : NO')

Structured answer keys              : 80 — hash VERIFIED
Rubric assignments                  : 640 — hash VERIFIED
Score-blind context rows            : 2,400 — hash VERIFIED
Cell 7B2 materialization protocol   : VERIFIED
Cell 7B2 adjudication protocol      : VERIFIED
Answer-key row content opened       : NO
Rubric row content opened           : NO
Alias-to-condition mapping opened   : NO


In [6]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C8_BLINDED_REVIEW_PACKET_MATERIALIZATION_ONLY_'
    'FROM_1440_FROZEN_STRUCTURED_RESPONSES_80_STRUCTURED_ANSWER_KEYS_640_RUBRIC_'
    'ASSIGNMENTS_AND_2400_SCORE_BLIND_CONTEXT_ROWS_TWO_INDEPENDENT_REVIEWERS_'
    'THIRD_ADJUDICATOR_NO_CONDITION_UNBLINDING_SCORE_BEARING_ARTIFACTS_RUN_AGGREGATION_'
    'RAG_METRICS_BOOTSTRAP_OR_ARM_COMPARISON'
)

protocol_snapshot = {
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'primary_questions': 80,
    'structured_answer_keys': 80,
    'rubric_assignments': 640,
    'rubric_dimensions_per_question': 8,
    'llm_response_observations': 1440,
    'independent_reviewers': 2,
    'third_adjudicator_for_disagreements': True,
    'atomic_factual_claims_are_scoring_unit': True,
    'primary_endpoint': 'fraction of atomic factual claims that are both correct and citation-supported',
    'reviewer_blinding': [
        'experimental condition identity',
        'GES scores',
        'metadata scores',
        'quality ranks',
    ],
    'egfr_role': 'exploratory; report separately',
    'answer_key_row_content_opened_in_cell_7c7': False,
    'rubric_row_content_opened_in_cell_7c7': False,
    'condition_unblinded': False,
    'rag_metrics_calculated': False,
}

authorization = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'authorization_decision': authorization_decision,
    'authorized_cell': {'cell_id': '7C8', 'authorized_once': True},
    'authorized_operations': [
        'Load exact Cell 7C6 V2 structured responses.',
        'Load exact Cell 7B3 structured answer keys.',
        'Load exact Cell 7B3 rubric assignments.',
        'Load exact Cell 7C4 score-blind context inventory.',
        'Materialize blinded reviewer packets for two independent reviewers.',
        'Materialize a third-adjudicator disagreement-resolution template.',
        'Preserve atomic factual claims as scoring unit and frozen primary endpoint.',
    ],
    'prohibited_operations': [
        'Load Cell 7A3 score rows.',
        'Load Cell 7C2 score-bearing reranking audit.',
        'Load or materialize alias-to-A/F condition mapping.',
        'Calculate condition/alias/arm-level scientific performance.',
        'Aggregate three LLM runs into a scientific endpoint.',
        'Calculate primary or secondary endpoints.',
        'Run bootstrap inference.',
        'Declare any condition superior/inferior.',
    ],
    'answer_key_access_authorized_in_cell_7c8': True,
    'rubric_access_authorized_in_cell_7c8': True,
    'score_blind_context_access_authorized_in_cell_7c8': True,
    'condition_unblinding_authorized': False,
    'score_bearing_artifact_access_authorized': False,
    'scientific_metric_calculation_authorized': False,
    'run_aggregation_authorized': False,
    'bootstrap_inference_authorized': False,
}

checks = OrderedDict([
    ('7c6_7_artifacts', sum(r['source_cell']=='7C6_V2' for r in verified_inputs)==7),
    ('7c6_1440_rows', parquet_rows(CELL_7C6['structured_outputs'][0])==1440),
    ('answer_80', answer_rows==80),
    ('rubric_640', rubric_rows==640),
    ('context_2400', context_rows==2400),
    ('answer_hash', sha256_file(answer_path)==CELL_7B3_ANSWER[1]),
    ('rubric_hash', sha256_file(rubric_path)==CELL_7B3_RUBRIC[1]),
    ('context_hash', sha256_file(context_path)==CELL_7C4_CONTEXT[1]),
    ('mat_protocol_hash', sha256_file(mat_protocol_path)==H_7B2_MATERIALIZATION),
    ('adj_protocol_hash', sha256_file(adj_protocol_path)==H_7B2_ADJUDICATION),
    ('answer_rows_not_opened', True),
    ('rubric_rows_not_opened', True),
    ('condition_mapping_not_opened', True),
    ('score_bearing_not_opened', True),
    ('metrics_not_calculated', True),
])
failed = [k for k,v in checks.items() if not bool(v)]
if failed:
    raise RuntimeError('Cell 7C7 checks failed:\\n- ' + '\\n- '.join(failed))

write_json(OUTPUTS['authorization'], authorization); write_sidecar(OUTPUTS['authorization'])
write_csv(OUTPUTS['input_inventory'], pd.DataFrame(verified_inputs)); write_sidecar(OUTPUTS['input_inventory'])
write_json(OUTPUTS['protocol_snapshot'], protocol_snapshot); write_sidecar(OUTPUTS['protocol_snapshot'])

terminal_decision = (
    'PASS_STAGE7C7_COMPLETE_CELL7C6_V2_GENERATION_PACKAGE_AND_FROZEN_CELL7B2_7B3_'
    'EVALUATION_DESIGN_REVERIFIED_CHECKSUM_PROTECTED_CELL7C8_BLINDED_REVIEW_PACKET_'
    'MATERIALIZATION_ONLY_AUTHORIZED_ANSWER_KEYS_AND_RUBRICS_MAY_BE_OPENED_IN_7C8_'
    'NO_CONDITION_UNBLINDING_SCORE_BEARING_ARTIFACTS_RUN_AGGREGATION_RAG_METRICS_'
    'BOOTSTRAP_OR_ARM_COMPARISON'
)

qc = {
    'cell_id': CELL_ID,
    'checks': {k: bool(v) for k,v in checks.items()},
    'passed_checks': len(checks),
    'failed_checks': 0,
    'total_checks': len(checks),
    'terminal_decision': terminal_decision,
}
write_json(OUTPUTS['qc'], qc); write_sidecar(OUTPUTS['qc'])

manifest = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c6_v2_manifest_sha256': CELL_7C6['manifest'][1],
        'cell_7c6_v2_structured_outputs_sha256': CELL_7C6['structured_outputs'][1],
        'cell_7b3_structured_answer_keys_sha256': CELL_7B3_ANSWER[1],
        'cell_7b3_rubric_assignments_sha256': CELL_7B3_RUBRIC[1],
        'cell_7b2_materialization_protocol_sha256': H_7B2_MATERIALIZATION,
        'cell_7b2_adjudication_protocol_sha256': H_7B2_ADJUDICATION,
        'cell_7c4_context_inventory_sha256': CELL_7C4_CONTEXT[1],
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C8',
    'answer_key_access_authorized_in_next_cell': True,
    'condition_unblinding_authorized': False,
    'scientific_metric_calculation_authorized': False,
    'bootstrap_inference_authorized': False,
}
write_json(OUTPUTS['manifest'], manifest); write_sidecar(OUTPUTS['manifest'])

for p in OUTPUTS.values():
    assert p.exists() and sidecar_is_valid(p)

auth_rb = load_json(OUTPUTS['authorization'])
qc_rb = load_json(OUTPUTS['qc'])
man_rb = load_json(OUTPUTS['manifest'])
prot_rb = load_json(OUTPUTS['protocol_snapshot'])

readback = OrderedDict([
    ('auth_exact', auth_rb['authorization_decision']==authorization_decision),
    ('next_7c8', man_rb['next_authorized_cell']=='7C8'),
    ('answer_access_true', auth_rb['answer_key_access_authorized_in_cell_7c8'] is True),
    ('unblinding_false', auth_rb['condition_unblinding_authorized'] is False),
    ('metrics_false', auth_rb['scientific_metric_calculation_authorized'] is False),
    ('qc_zero', int(qc_rb['failed_checks'])==0),
    ('primary_endpoint_exact', prot_rb['primary_endpoint']=='fraction of atomic factual claims that are both correct and citation-supported'),
    ('all_sidecars', all(sidecar_is_valid(p) for p in OUTPUTS.values())),
])
failed2 = [k for k,v in readback.items() if not bool(v)]
if failed2:
    raise RuntimeError('Cell 7C7 readback failed:\\n- ' + '\\n- '.join(failed2))

total = len(checks)+len(readback)
sep='='*150
print('\\n'+sep)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C7')
print('BLINDED EVALUATION-PACKET MATERIALIZATION AUTHORIZATION')
print(sep)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')
print('\\nUPSTREAM CELL 7C6 V2 REVERIFICATION')
print(f'Cell 7C6 V2 manifest SHA-256                  : {CELL_7C6["manifest"][1]}')
print('Cell 7C6 V2 terminal PASS verified            : YES')
print('Frozen Cell 7C6 V2 artifacts                  : 7/7 exact hashes + sidecars')
print('Structured-valid LLM responses                : 1,440 / 1,440')
print('\\nFROZEN EVALUATION DESIGN')
print('Structured answer keys                        : 80 — hash VERIFIED')
print('Question-rubric assignments                   : 640 — hash VERIFIED')
print('Rubric dimensions per question                : 8')
print('Score-blind context rows                      : 2,400 — hash VERIFIED')
print('Independent reviewers                         : 2')
print('Third adjudicator                             : YES')
print('Atomic factual claims                         : scoring unit')
print('Primary endpoint                              : correct AND citation-supported atomic claims')
print('EGFR                                          : exploratory; report separately')
print('\\nCELL 7C8 AUTHORIZATION')
print('Answer-key row access                         : AUTHORIZED')
print('Rubric row access                             : AUTHORIZED')
print('Score-blind context row access                : AUTHORIZED')
print('Blinded reviewer packet materialization       : AUTHORIZED')
print('Condition identity unblinding                 : PROHIBITED')
print('Score-bearing artifacts                       : PROHIBITED')
print('Run aggregation / RAG metrics                 : PROHIBITED')
print('Bootstrap / arm comparison                    : PROHIBITED')
print('\\nCELL 7C7 FROZEN OUTPUTS')
for label,path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')
print(f'\\nQC checks                                      : {total}/{total} PASS')
print('\\nFINAL DECISION                                : '+terminal_decision)
print(sep)

\n======================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C7
BLINDED EVALUATION-PACKET MATERIALIZATION AUTHORIZATION
Notebook                                      : 14_GES_Aware_Genomic_RAG_Cell_7C7_Blinded_Evaluation_Packet_Authorization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM CELL 7C6 V2 REVERIFICATION
Cell 7C6 V2 manifest SHA-256                  : 9a84701858829d650d826109e32b3708f4b0c5d7b2ab393dc3f3d3afd5cc8293
Cell 7C6 V2 terminal PASS verified            : YES
Frozen Cell 7C6 V2 artifacts                  : 7/7 exact hashes + sidecars
Structured-valid LLM responses                : 1,440 / 1,440
\nFROZEN EVALUATION DESIGN
Structured answer keys                        : 80 — hash VERIFIED
Question-rubric assignments                   : 640 — hash VERIFIED
Rubric dimensions per question 